In [ ]:
# pip install pandas deep-translator tqdm

import pandas as pd
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ======================================
# BACA FILE CSV
# ======================================

INPUT_FILE = "chunk_5.csv"
OUTPUT_FILE = "chunk_5_indonesia.csv"

df = pd.read_csv(INPUT_FILE)

# ======================================
# CEK KOLOM
# ======================================

print("Kolom tersedia:")
print(df.columns.tolist())

# ======================================
# INISIALISASI TRANSLATOR
# ======================================

translator = GoogleTranslator(
    source="en",
    target="id"
)

# ======================================
# FUNGSI TRANSLASI
# ======================================

def translate_text(text):
    try:
        if pd.isna(text):
            return ""

        text = str(text).strip()

        if text == "":
            return ""

        return translator.translate(text)

    except Exception as e:
        print(f"Error: {text[:50]} --> {e}")
        return text

# ======================================
# AMBIL DATA YANG AKAN DITERJEMAHKAN
# ======================================

titles = df["News Title"].tolist()

# ======================================
# MULTITHREAD TRANSLATION
# ======================================
# Ubah max_workers sesuai kebutuhan:
# 10  = aman
# 20  = cepat
# 30+ = berisiko kena rate limit

MAX_WORKERS = 20

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    translated_titles = list(
        tqdm(
            executor.map(translate_text, titles),
            total=len(titles),
            desc="Menerjemahkan"
        )
    )

# ======================================
# SIMPAN HASIL
# ======================================

df["News Title Indonesia"] = translated_titles

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nTranslasi selesai!")
print(f"Hasil disimpan ke: {OUTPUT_FILE}")
print(df.head())